# Illustrative aluminum evaporation from legacy coefficients

## Evidence status

The Antoine coefficients and validity interval below are preserved from legacy project notebooks, which attributed them to the NIST Chemistry WebBook and Stull (1947). That attribution has **not** been independently verified during this migration. The coefficients are therefore treated as unverified legacy inputs, not literature-verified reference data.

## Objectives

1. Express the retained Antoine convention through the maintained unit-aware model.
2. Convert the package's aluminum molar-mass entry to one-particle mass with explicit units.
3. Evaluate the planar three-dimensional Hertz--Knudsen evaporation limit with zero ambient pressure.
4. Plot equilibrium pressure and mass flux while preserving the provisional evidence status.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from projectkoios.physkit.constants import SI
from projectkoios.physkit.elements import ELEMENTS
from projectkoios.physkit.thermal.thermo.vaporpressure.antoine import (
    AntoineTemperatureRange,
    AntoineVaporPressureModel,
)
from projectkoios.physkit.thermal.transport.phase_change.hertz_knudsen3d import (
    PlanarHertzKnudsen3DNetFluxModel,
)
from projectkoios.physkit.units import (
    PhysicalUnit,
    ScalarQuantity,
    Unitless,
    VectorQuantity,
)

In [ ]:
temperature_unit = PhysicalUnit(expression="kelvin")
pressure_unit = PhysicalUnit(expression="pascal")
minimum_temperature_kelvin = 1557.0
maximum_temperature_kelvin = 2329.0
temperature_sample_count = 500
temperatures = VectorQuantity(
    magnitude=np.linspace(
        minimum_temperature_kelvin,
        maximum_temperature_kelvin,
        temperature_sample_count,
        dtype=np.float64,
    ),
    unit=temperature_unit,
)
aluminum_vapor_pressure_model = AntoineVaporPressureModel(
    A=ScalarQuantity(magnitude=5.73623, unit=Unitless()),
    B=ScalarQuantity(magnitude=13204.109, unit=temperature_unit),
    C=ScalarQuantity(magnitude=-24.306, unit=temperature_unit),
    coefficient_temperature_unit=temperature_unit,
    coefficient_pressure_unit=PhysicalUnit(expression="bar"),
    valid_temperature_range=AntoineTemperatureRange(
        lower=ScalarQuantity(
            magnitude=minimum_temperature_kelvin,
            unit=temperature_unit,
        ),
        upper=ScalarQuantity(
            magnitude=maximum_temperature_kelvin,
            unit=temperature_unit,
        ),
    ),
    source=("Unverified legacy attribution: NIST Chemistry WebBook; Stull (1947)."),
)
pressure_evaluation = aluminum_vapor_pressure_model.evaluate(
    temperatures=temperatures,
    pressure_unit=pressure_unit,
)

In [ ]:
aluminum_molar_mass_gram_per_mole = float(ELEMENTS["Al"].mass)
aluminum_particle_mass = ScalarQuantity(
    magnitude=(aluminum_molar_mass_gram_per_mole * 1.0e-3 / SI.N_A),
    unit=PhysicalUnit(expression="kilogram"),
)
flux_model = PlanarHertzKnudsen3DNetFluxModel(
    particle_mass=aluminum_particle_mass,
    evaporation_coefficient=ScalarQuantity(magnitude=1.0, unit=Unitless()),
    condensation_coefficient=ScalarQuantity(magnitude=0.0, unit=Unitless()),
)
flux_evaluation = flux_model.evaluate(
    temperatures=temperatures,
    equilibrium_pressures=pressure_evaluation.pressures,
    ambient_pressures=VectorQuantity(
        magnitude=np.zeros(temperature_sample_count, dtype=np.float64),
        unit=pressure_unit,
    ),
    number_flux_unit=PhysicalUnit(expression="meter ** -2 / second"),
    mass_flux_unit=PhysicalUnit(expression="kilogram / meter ** 2 / second"),
)
assert np.all(np.diff(pressure_evaluation.pressures.magnitude) > 0.0)
assert np.all(np.diff(flux_evaluation.net_mass_flux.magnitude) > 0.0)
assert np.all(flux_evaluation.net_number_flux.magnitude > 0.0)

In [ ]:
figure_size_inches = (11.0, 4.5)
figure, (pressure_axis, flux_axis) = plt.subplots(
    nrows=1,
    ncols=2,
    figsize=figure_size_inches,
)
pressure_axis.semilogy(
    temperatures.magnitude,
    pressure_evaluation.pressures.magnitude,
    label="legacy-coefficient equilibrium pressure",
)
pressure_axis.set(
    xlabel="temperature (K)",
    ylabel="pressure (Pa)",
    title="Illustrative aluminum vapor pressure",
)
pressure_axis.grid(True)
pressure_axis.legend()

flux_axis.semilogy(
    temperatures.magnitude,
    flux_evaluation.net_mass_flux.magnitude,
    label="zero-background mass flux",
)
flux_axis.set(
    xlabel="temperature (K)",
    ylabel="mass flux (kg m⁻² s⁻¹)",
    title="Illustrative evaporation limit",
)
flux_axis.grid(True)
flux_axis.legend()
figure.tight_layout()
plt.show()

## Interpretation

For the retained coefficient set, equilibrium pressure and the corresponding zero-background evaporation flux increase over the represented temperature interval. These are calculated outputs of the identified software model using unverified legacy coefficients. They are not independently validated aluminum vapor-pressure or deposition-rate results.

## Exercises

1. Verify the coefficient source and exact unit convention against a primary reference before using the values as literature evidence.
2. Add a nonzero ambient aluminum partial pressure and inspect the signed net flux.
3. Vary the evaporation and condensation coefficients separately.
4. Propagate documented coefficient uncertainty only after a verified source supplies it.
